# 第15章 自分のコードに本物のGPT-2を入れる

OpenAIが公開した1億2400万個の数を、自作GPTに流し込む

教科書のページ：https://sakunokyumei.github.io/llm-research-textbook/15-gpt2.html

上のセルから順に、再生ボタン（▶）か `Shift + Enter` で実行します。説明は教科書のページにあります。


## GPT-2をダウンロードする


In [ ]:
import math
import torch
import torch.nn as nn
import torch.nn.functional as F
from transformers import GPT2LMHeadModel, AutoTokenizer

hf = GPT2LMHeadModel.from_pretrained("gpt2")
tok = AutoTokenizer.from_pretrained("gpt2")
hf.eval()
cfg = hf.config
print("層の数：", cfg.n_layer, " ヘッド：", cfg.n_head, " 幅：", cfg.n_embd)
print("語彙：", cfg.vocab_size, " 窓の大きさ：", cfg.n_positions)
print(f"パラメータの数：{sum(p.numel() for p in hf.parameters()):,}")

## 重みの名前と形を見る


In [ ]:
sd_hf = hf.state_dict()
for k in list(sd_hf.keys())[:16]:
    print(f"{k:42s} {tuple(sd_hf[k].shape)}")

## GPT-2と同じ形のGPTを書く


In [ ]:
class CausalSelfAttention(nn.Module):
    def __init__(self, n_embd, n_head, block_size):
        super().__init__()
        self.n_head = n_head
        self.c_attn = nn.Linear(n_embd, 3 * n_embd)     # Q・K・Vをまとめて作る
        self.c_proj = nn.Linear(n_embd, n_embd)
        self.register_buffer("tril", torch.tril(torch.ones(block_size, block_size)), persistent=False)

    def forward(self, x):
        B, T, C = x.shape
        hs = C // self.n_head                           # 1ヘッドの大きさ
        q, k, v = self.c_attn(x).split(C, dim=2)        # 最後の軸をC個ずつに切る（2304→768・768・768）。文字列のsplitとは別物
        q = q.view(B, T, self.n_head, hs).transpose(1, 2)   # [B, ヘッド, T, hs]
        k = k.view(B, T, self.n_head, hs).transpose(1, 2)
        v = v.view(B, T, self.n_head, hs).transpose(1, 2)
        att = q @ k.transpose(-2, -1) / math.sqrt(hs)   # 全ヘッドの点数を一度に
        att = att.masked_fill(self.tril[:T, :T] == 0, float("-inf"))
        att = F.softmax(att, dim=-1)
        y = att @ v                                     # [B, ヘッド, T, hs]
        y = y.transpose(1, 2).contiguous().view(B, T, C)    # ヘッドを横につなげ直す
        return self.c_proj(y)


class MLP(nn.Module):
    def __init__(self, n_embd):
        super().__init__()
        self.c_fc = nn.Linear(n_embd, 4 * n_embd)
        self.gelu = nn.GELU(approximate="tanh")         # GPT-2のGELU（近似版）
        self.c_proj = nn.Linear(4 * n_embd, n_embd)

    def forward(self, x):
        return self.c_proj(self.gelu(self.c_fc(x)))


class Block(nn.Module):
    def __init__(self, n_embd, n_head, block_size):
        super().__init__()
        self.ln_1 = nn.LayerNorm(n_embd)
        self.attn = CausalSelfAttention(n_embd, n_head, block_size)
        self.ln_2 = nn.LayerNorm(n_embd)
        self.mlp = MLP(n_embd)

    def forward(self, x):
        x = x + self.attn(self.ln_1(x))
        x = x + self.mlp(self.ln_2(x))
        return x


class GPT2(nn.Module):
    def __init__(self, vocab_size, block_size, n_embd, n_head, n_layer):
        super().__init__()
        self.block_size = block_size
        self.transformer = nn.ModuleDict(dict(
            wte=nn.Embedding(vocab_size, n_embd),
            wpe=nn.Embedding(block_size, n_embd),
            h=nn.ModuleList([Block(n_embd, n_head, block_size) for _ in range(n_layer)]),
            ln_f=nn.LayerNorm(n_embd),
        ))
        self.lm_head = nn.Linear(n_embd, vocab_size, bias=False)
        self.lm_head.weight = self.transformer.wte.weight    # 重み共有（weight tying）

    def forward(self, idx):
        B, T = idx.shape
        x = self.transformer.wte(idx) + self.transformer.wpe(torch.arange(T, device=idx.device))
        for block in self.transformer.h:
            x = block(x)
        return self.lm_head(self.transformer.ln_f(x))

## 重みを流し込む


In [ ]:
mine = GPT2(cfg.vocab_size, cfg.n_positions, cfg.n_embd, cfg.n_head, cfg.n_layer)
mine.eval()
sd = mine.state_dict()
transposed = ("attn.c_attn.weight", "attn.c_proj.weight", "mlp.c_fc.weight", "mlp.c_proj.weight")

copied = 0
with torch.no_grad():
    for k in sd:
        src = sd_hf[k].t() if k.endswith(transposed) else sd_hf[k]
        assert sd[k].shape == src.shape, (k, sd[k].shape, src.shape)
        sd[k].copy_(src)
        copied += 1
print("コピーした重みの数：", copied)

## 答え合わせ


In [ ]:
ids = tok("Hello, I'm a language model,", return_tensors="pt").input_ids
with torch.no_grad():
    logits_mine = mine(ids)
    logits_hf = hf(ids).logits
print("出力の形：", tuple(logits_mine.shape))
print("差は0.001未満？", (logits_mine - logits_hf).abs().max().item() < 1e-3)
print("一致した？", torch.allclose(logits_mine, logits_hf, atol=1e-3))

## GPT-2に文章を書かせる


In [ ]:
@torch.no_grad()
def generate(model, text, n=40, temperature=0.8, top_k=50, seed=0):
    torch.manual_seed(seed)
    idx = tok(text, return_tensors="pt").input_ids
    for _ in range(n):
        logits = model(idx[:, -model.block_size:])[:, -1, :] / temperature
        v, _ = torch.topk(logits, top_k)
        logits[logits < v[:, [-1]]] = -float("inf")    # 上位k個以外は候補から外す
        next_id = torch.multinomial(F.softmax(logits, dim=-1), 1)
        idx = torch.cat([idx, next_id], dim=1)
    return tok.decode(idx[0])

print(generate(mine, "Hello, I'm a language model,"))

In [ ]:
print(generate(mine, "吾輩は猫である。"))

## 事前学習：GPT-2はどうやって作られたか


In [ ]:
def train_flops(n_params, n_tokens):
    return 6 * n_params * n_tokens          # 学習の計算量 ≈ 6 × N × D

def train_memory_gb(n_params):
    return n_params * 16 / 1e9              # AdamWで学習するときのメモリの目安（1パラメータ16バイト）

models = {"漱石GPT": (1.75e6, 2e7), "GPT-2 small": (124e6, 9e9),
          "Qwen2.5-0.5B": (0.49e9, 18e12), "70Bのモデル": (70e9, 15e12)}
for name, (N, D) in models.items():
    print(f"{name:12s} 計算量 {train_flops(N, D):9.1e} 回  学習メモリ {train_memory_gb(N):9.2f} GB"
          f"  Chinchilla推奨データ {20 * N:8.1e} トークン")